In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")


In [9]:
from langchain_community import document_loaders
from langchain_community.document_loaders import TextLoader
loader=TextLoader('my_document.txt')
document=loader.load()

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
documents_chunk=splitter.split_documents(document)

In [13]:
from huggingface_hub import snapshot_download
MODEL_PATH = "./models/all-MiniLM-L6-v2"

# Only download if not already present
if not os.path.exists(MODEL_PATH):
    snapshot_download(
        repo_id="sentence-transformers/all-MiniLM-L6-v2",
        local_dir=MODEL_PATH
    )

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3650.98it/s]


In [15]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(documents_chunk,embeddings)

In [16]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [23]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [22]:
llm=ChatGroq(model="llama3-8b-8192",api_key=groq_api_key)


In [25]:
from langchain_core.tracers import context
from langchain_core.messages import content

#prompt
prompt=ChatPromptTemplate.from_template(
    """ 
    Answer the question based on the context below only.
    If you don't know, say you don't know.

    context:{context}
    Question:{Question}
    """
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#Chain

chain=(
    {
        "context":retriever | format_docs,
    "question":RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser

)


response=chain.invoke("How many spiderman movies are there till date?")
print(response)

KeyError: "Input to ChatPromptTemplate is missing variables {'Question'}.  Expected: ['Question', 'context'] Received: ['context', 'question']\nNote: if you intended {Question} to be part of the string and not a variable, please escape it with double curly braces like: '{{Question}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "